# Group

Name: Slumdogs

Memebrs:
 - Alp Tuna alp.tuna@mail.polimi.it
 - Çisem Özden cisem.ozden@mail.polimi.it
 - Fidel Guri fidel.guri@mail.polimi.it
 - Matteo Enut matteo.enut@mail.polimi.it
 - Vasileios Petrakopoulos vasileios.petrakopoulos@mail.polimi.it

 Link to video: https://drive.google.com/file/d/1GOg-UoyeftFjOu9Fb6s4WjTb910qSIVo/view?usp=sharing

# Libraries

In [ ]:
!pip -q install transformers accelerate bitsandbytes sentencepiece
!pip -q install trafilatura googlenewsdecoder sentence-transformers
!pip -q install mistral-common --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.4/300.4 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 92.8 MB/s eta 0:00:00


In [ ]:
# this might be needed depending on chosen model and environment
# uncomment if you see relevant error and restart kernel after executing this cell (crucial)
#!pip -q install "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main"

In [ ]:
# All third-party / stdlib imports used throughout the notebook live here.
import os
import re
import sys
import io
import time
import json
import math
import bisect
import signal
import shutil
import hashlib
import contextlib
import statistics
import itertools
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, replace
from typing import Callable, Union, Optional, List
from urllib.parse import quote

import requests
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
    Mistral3ForConditionalGeneration,
)
from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from huggingface_hub import login

# RAG: Wikipedia dump access, news fetching, cross-encoder reranking
import sqlite3
import html
import xml.etree.ElementTree as ET
import concurrent.futures
import pyarrow.parquet as pq
import trafilatura
from googlenewsdecoder import gnewsdecoder
from sentence_transformers import CrossEncoder


# Kaggle secrets
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()


In [ ]:
# Fetch the secret from Kaggle's vault
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
# Log in to Hugging Face
login(token=hf_token)

In [ ]:
# polimillionaire client

# Define the path to the directory containing your package
package_parent_dir = '/kaggle/input/datasets/vaspetrak/millionaire-client/millionaire_client'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

from millionaire_client import MillionaireClient, AuthenticationError
pwd = user_secrets.get_secret("poli-millionaire")

API_URL = "http://131.175.15.22:51111/"
username = "vpetrak"
password = pwd

client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")

['/kaggle/working', '/kaggle/lib/kagglegym', '/kaggle/lib', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/kaggle/input/datasets/vaspetrak/millionaire-client/millionaire_client']

Welcome, vpetrak! (Role: student)


# Model Configuration

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"

# The below models work for our project without any fine tuning
# Best-performing models tested so far marked with ★
# Qwen:    "Qwen/Qwen2.5-14B-Instruct" ★, "Qwen/Qwen2.5-7B-Instruct", "Qwen/Qwen3-4B-Instruct-2507"
# Mistral: "mistralai/Mistral-Small-3.2-24B-Instruct-2506" ★, "mistralai/Mistral-7B-Instruct-v0.3", "mistralai/Ministral-3-8B-Instruct-2512"
# Gemma:   "google/gemma-4-E4B-it" ★, "google/gemma-2-9b-it", "google/gemma-3-4b-it", "google/gemma-3-12b-it"
# Llama:   "meta-llama/Llama-3.1-8B-Instruct", "meta-llama/Llama-3.2-3B-Instruct"
# Phi:     "microsoft/phi-4"

GENERATION_CONFIG = {
    "max_new_tokens": 8,           # number is enough; reduces latency
    "do_sample": False,            # greedy; deterministic for fair comparisons
    "use_cache": True,             # set to True if model fits memory easily but is slow
    "output_scores": True,         # logits at every gen step (for confidence)
    "return_dict_in_generate": True,
}

USE_QUANTIZATION = True            # set to true if model very large

# Latency we consider "risky" given the 30s server timeout
LATENCY_DANGER_THRESHOLD_S = 20.0

## Base Model Comparison (no RAG, no CoT)

Per-category accuracy and mean latency per question across three candidate models (comparison took place before the News and Philosophy categories were added).

| Model | Entertainment | Ancient History & Politics | Science & Nature | Maths
|---|---|---|---|---|
| Qwen/Qwen2.5-14B-Instruct | 84.0% / 1.13s | 81.0% / 1.18s | 90.0% / 1.18s | 67.0% / 1.22s |
| google/gemma-4-E4B-it | 86.0% / 0.29s | 79.0% / 0.29s | 93.0% / 0.30s | 50.0% / 0.31s |
| mistralai/Mistral-Small-3.2-24B-Instruct-2506 | 80.0% / 1.18s | 80.0% / 1.15s | 91.0% / 1.15s | 49.0% / 1.23s |

*Accuracy % / mean latency per question (s).*

In [ ]:
# model family helpers

def is_gemma3(model_name):

    model_name = model_name.lower()

    return "gemma-3" in model_name


def is_gemma2(model_name):

    model_name = model_name.lower()

    return "gemma-2" in model_name


def is_mistral3(model_name):

    model_name = model_name.lower()

    return "mistral-small-3" in model_name or "magistral" in model_name

In [ ]:
# quantization (4-bit) and model args

if USE_QUANTIZATION:

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    model_kwargs = {
        "quantization_config": bnb_config,
        "device_map": "auto",
        "dtype": torch.float16,
    }

else:

    model_kwargs = {
        "device_map": "auto",
        "dtype": torch.float16,
    }

In [ ]:
# load/swap model (and tokenizer where needed)

# paths for models mounted via Kaggle Models / Datasets
# set to None (or omit) for models we want to always fetch from HF hub
KAGGLE_MODEL_PATHS = {
    "Qwen/Qwen2.5-14B-Instruct":
        "/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct/1",
    "google/gemma-4-E4B-it":
        "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1",
}


def _resolve_model_path(model_name):
    """
    Return (path_or_name, source_label).
    Prefers a Kaggle-mounted local copy when present; otherwise returns the
    HF hub name so transformers will download (and cache) it.
    """
    local = KAGGLE_MODEL_PATHS.get(model_name)
    if local and Path(local).exists():
        return local, "kaggle"
    return model_name, "hub"


def load_model_globally(model_name):
    """
    Load (or swap to) a model.
    - Frees GPU memory and globals of any currently loaded model.
    - Uses /kaggle/input/... when available, else HF hub.
    - HF hub uses its on-disk cache automatically, so a previously-downloaded
      model is reloaded for free on a later swap-back (no re-download).
    """
    global MODEL_NAME, model, tokenizer, processor, mistral_tokenizer
    import gc

    # Free GPU memory + globals of whatever was loaded before
    for var in ("model", "tokenizer", "processor", "mistral_tokenizer"):
        if var in globals() and globals()[var] is not None:
            globals()[var] = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    MODEL_NAME = model_name
    source_path, source = _resolve_model_path(model_name)
    print(f"Loading {model_name}  (source: {source}, path: {source_path})")

    if is_mistral3(MODEL_NAME):
        # Mistral's chat tokenizer is separate. If we have a local copy,
        # try loading the tokenizer file from it; else fall back to hub.
        mistral_tokenizer = None
        if source == "kaggle":
            for tk_name in ("tekken.json", "tokenizer.json"):
                tk_path = Path(source_path) / tk_name
                if tk_path.exists():
                    mistral_tokenizer = MistralTokenizer.from_file(str(tk_path))
                    break
        if mistral_tokenizer is None:
            mistral_tokenizer = MistralTokenizer.from_hf_hub(MODEL_NAME)

        model = Mistral3ForConditionalGeneration.from_pretrained(
            source_path, **model_kwargs
        ).eval()
        tokenizer = None
        processor = None

    elif is_gemma3(MODEL_NAME):
        processor = AutoProcessor.from_pretrained(source_path)
        model = Gemma3ForConditionalGeneration.from_pretrained(
            source_path, **model_kwargs
        ).eval()
        tokenizer = None
        mistral_tokenizer = None

    else:
        tokenizer = AutoTokenizer.from_pretrained(source_path, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(
            source_path, trust_remote_code=True, **model_kwargs
        )
        processor = None
        mistral_tokenizer = None

    print(f"  loaded: {model_name}")
    return model

In [ ]:
model = load_model_globally(MODEL_NAME)

Loading Qwen/Qwen2.5-14B-Instruct  (source: kaggle, path: /kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct/1)


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

  loaded: Qwen/Qwen2.5-14B-Instruct


# Prompt Engineering

In [ ]:
@dataclass
class PromptStrategy:
    """
    Bundles everything needed to evaluate a prompting approach.
    """
    name: str
    system_prompt: Union[str, Callable[[Optional[int]], str]]
    user_builder: Callable[[str, List[str], Optional[int]], str]
    parser: Callable[[str, dict], Optional[int]]
    max_new_tokens: int = 8
    n_samples: int = 1
    do_sample: bool = False
    temperature: float = 0.0
    use_rag: bool = False

    def get_system(self, competition_id=None):
        if callable(self.system_prompt):
            return self.system_prompt(competition_id)
        return self.system_prompt


# Parsers

def parse_direct_digit(raw_output, confidence):
    """
    For direct-answer prompts. Regex for first 1-4, fallback to logit argmax.
    """
    m = re.search(r"[1-4]", raw_output.strip())
    if m:
        return int(m.group())
    if confidence:
        return max(confidence, key=confidence.get)
    return None


def parse_answer_line(raw_output, confidence):
    """
    For CoT-style outputs. Prefers explicit 'Answer: N' markers.
    """
    text = raw_output.strip()
    explicit_patterns = [
        r"(?:final\s+answer|the\s+answer\s+is|answer)\s*[:\-]?\s*([1-4])",
        r"(?:i\s+(?:choose|pick|select))\s+(?:option\s+)?([1-4])",
        r"option\s+([1-4])",
    ]
    for pat in explicit_patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return int(m.group(1))
    # Bare digit at end of text
    m = re.search(r"([1-4])\s*[\.\)]?\s*$", text)
    if m:
        return int(m.group(1))
    # Last digit anywhere
    digits = re.findall(r"[1-4]", text)
    if digits:
        return int(digits[-1])
    if confidence:
        return max(confidence, key=confidence.get)
    return None


def parse_eliminate(raw_output, confidence):
    """
    For P5 elimination format: 'N: yes/no'. Falls through to parse_answer_line.
    """
    text = raw_output.strip()
    # Explicit Answer: line wins
    m = re.search(r"answer\s*[:\-]?\s*([1-4])", text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    # Affirmative marker on a numbered option
    positive = r"(?:yes|correct|possible|right|true|likely|plausible)"
    m = re.search(rf"([1-4])\s*[:.\-]\s*{positive}\b", text, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return parse_answer_line(text, confidence)


# User-prompt builders

def _user_builder_basic(question, options, competition_id=None):
    return (
        f"Question:\n{question}\n\n"
        f"Options:\n"
        f"1. {options[0]}\n"
        f"2. {options[1]}\n"
        f"3. {options[2]}\n"
        f"4. {options[3]}\n\n"
        f"Answer:"
    )


# P0: direct-answer baseline

P0_BASELINE = PromptStrategy(
    name="P0_baseline",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "You will receive one question and four numbered options.\n"
        "Choose the single most likely correct option.\n\n"
        "Output format:\n"
        "- Reply with ONLY one digit: 1, 2, 3, or 4.\n"
        "- No words, no punctuation, no explanation.\n"
        "- If unsure, still pick your best guess. You cannot skip."
    ),
    user_builder=_user_builder_basic,
    parser=parse_direct_digit,
    max_new_tokens=8,
)


# P1: few-shot

P1_FEWSHOT = PromptStrategy(
    name="P1_fewshot",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n"
        "Reply with ONLY one digit (1, 2, 3, or 4). No explanation.\n\n"
        "Examples:\n\n"
        "Question: What is the chemical symbol for gold?\n"
        "Options:\n1. Au\n2. Ag\n3. Go\n4. Gd\nAnswer: 1\n\n"
        "Question: What is 12 multiplied by 7?\n"
        "Options:\n1. 72\n2. 84\n3. 96\n4. 108\nAnswer: 2\n\n"
        "Question: What is the largest planet in our solar system?\n"
        "Options:\n1. Earth\n2. Mars\n3. Jupiter\n4. Saturn\nAnswer: 3\n\n"
        "Question: In which year did World War II end?\n"
        "Options:\n1. 1943\n2. 1944\n3. 1946\n4. 1945\nAnswer: 4\n"
    ),
    user_builder=_user_builder_basic,
    parser=parse_direct_digit,
    max_new_tokens=8,
)


# P2: chain-of-thought

P2_COT = PromptStrategy(
    name="P2_cot",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "Think briefly about which option is correct — at most 3 short sentences.\n"
        "Then on the final line write exactly:\n"
        "Answer: N\n"
        "where N is 1, 2, 3, or 4."
    ),
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=128,
)


# P3: CoT + self-consistency

P3_SELF_CONSISTENCY = PromptStrategy(
    name="P3_self_consistency",
    system_prompt=P2_COT.system_prompt,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=128,
    n_samples=3,           # increase to 5 if accuracy boost justifies the latency
    do_sample=True,
    temperature=0.7,
)


# P4: per-competition specialization

_P4_SYSTEMS = {
    0: (  # Entertainment
        "You are an expert on entertainment trivia (movies, music, TV, "
        "games, books, celebrities). Recall the precise fact and reply "
        "with ONLY one digit (1, 2, 3, or 4)."
    ),
    1: (  # Ancient History and Politics
        "You are an expert on ancient history and politics. Consider "
        "dates, historical context, and political systems carefully. "
        "Reply with ONLY one digit (1, 2, 3, or 4)."
    ),
    2: (  # Science and Nature
        "You are an expert on science and nature. Apply established "
        "scientific principles and verified knowledge. Reply with ONLY "
        "one digit (1, 2, 3, or 4)."
    ),
    3: (  # Maths
        "You are an expert at multiple-choice mathematics problems.\n"
        "Work the problem step by step in 2-4 short lines.\n"
        "Then on the final line write exactly:\nAnswer: N\n"
        "where N is 1, 2, 3, or 4."
    ),
    4: (  # Philosophy and Psychology
        "You are an expert on philosophy and psychology. Consider "
        "conceptual frameworks, theoretical positions, and established "
        "psychological findings. Reply with ONLY one digit (1, 2, 3, or 4)."
    ),
    5: (  # News
        "You are an expert on current events and recent news. "
        "If reference passages are provided, rely on them — they may "
        "contain information that postdates your training. Reply with "
        "ONLY one digit (1, 2, 3, or 4)."
    ),
}

def _p4_system(competition_id):
    return _P4_SYSTEMS.get(competition_id, P0_BASELINE.system_prompt)

P4_SPECIALIZED = PromptStrategy(
    name="P4_specialized",
    system_prompt=_p4_system,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,           # handles both direct and CoT outputs
    max_new_tokens=128,                 # high enough for Maths CoT; others stop at EOS
)


# P5: eliminate impossible options

P5_ELIMINATE = PromptStrategy(
    name="P5_eliminate",
    system_prompt=(
        "You are a multiple-choice trivia expert.\n\n"
        "For each option, very briefly note whether it could be correct.\n"
        "Use the format 'N: yes/no — short reason' on one line per option.\n"
        "Then on the final line write exactly:\n"
        "Answer: N\n"
        "where N is 1, 2, 3, or 4.\n\n"
        "Example:\n"
        "1: no — wrong century\n"
        "2: yes — matches the description\n"
        "3: no — refers to a different event\n"
        "4: no — geographic mismatch\n"
        "Answer: 2"
    ),
    user_builder=_user_builder_basic,
    parser=parse_eliminate,
    max_new_tokens=200,
)


ALL_STRATEGIES = [P0_BASELINE, P1_FEWSHOT, P2_COT, P3_SELF_CONSISTENCY, P4_SPECIALIZED, P5_ELIMINATE]
# RAG-augmented strategy variants — identical to the originals but with use_rag=True.
P0_RAG = replace(P0_BASELINE, name="P0_baseline_rag", use_rag=True)
P2_RAG = replace(P2_COT,      name="P2_cot_rag",      use_rag=True)
P4_RAG = replace(P4_SPECIALIZED, name="P4_specialized_rag", use_rag=True)
RAG_STRATEGIES = [P0_RAG, P2_RAG, P4_RAG]


# Maths chain-of-thought: let the model reason briefly, then commit to a single
# "Answer: N" line. The ablation showed this beats the Python-sandbox tool path
# on competition 3 (~0.75 vs ~0.70). Token budget is tight so the hard cases
# still commit before the 30s wall.
_MATHS_COT_SYSTEM = (
    "You are an expert at multiple-choice mathematics.\n"
    "Reason in AT MOST 3 very short lines. Be terse.\n"
    "CRITICAL: your reasoning budget is limited, so you MUST commit to an "
    "answer quickly. As soon as you can, and ALWAYS before you run out of "
    "room, write the final line exactly as:\n"
    "Answer: N\n"
    "where N is 1, 2, 3, or 4. The 'Answer:' line is mandatory; never end "
    "without it. Output nothing after it."
)

P_MATHS_COT = PromptStrategy(
    name="P_maths_cot",
    system_prompt=_MATHS_COT_SYSTEM,
    user_builder=_user_builder_basic,
    parser=parse_answer_line,
    max_new_tokens=160,        # tight cap forces an early commit under the 30s wall
)

# RAG (Retrieval-Augmented Generation)

Two retrievers survive the investigation; the dead ends (DuckDuckGo, Wikidata,
Wikinews, offline BM25, the live opensearch API, and cross-encoder reranking of
whole articles) live in the report, not the code.

- **Wikipedia (offline, full article)** — for Entertainment (0), Ancient
  History (1) and Science (2). The LLM names article titles, we resolve them against an offline
  title->location map and read the text straight from the Kaggle dump (no network),
  then feed the **whole article** to the model. Beat every reranked-chunk variant.
- **Google News (full article bodies)** — for News (5), the one clear significant
  win (+~10pp, p < 0.001). Google News ranking supplies the recency the model lacks.

Maths (3), and Philosophy (4) use no RAG. Each retriever exposes a
`use_reranker` toggle (default `False` = full text, the keeper; `True` = reranked
chunks, kept for comparison but not routed).

In [ ]:
# Offline Wikipedia title -> (parquet file, row index) map.
# One-time build (~3-5 min, ~200 MB). The hybrid retriever uses this to pull
# full article text straight from the Kaggle dump with no API rate limit.
import sqlite3, os, time
import pyarrow.parquet as pq
from pathlib import Path

LOC_DB     = "/kaggle/working/wiki_loc.db"
ENWIKI_DIR = "/kaggle/input/datasets/organizations/wikimedia-foundation/wikipedia-structured-contents/enwiki/data"

if not Path(LOC_DB).exists():
    print("Building title->location map ...")
    t0 = time.time()
    conn = sqlite3.connect(LOC_DB)
    conn.execute("PRAGMA journal_mode=OFF")
    conn.execute("PRAGMA synchronous=OFF")
    conn.execute("CREATE TABLE wiki_loc (name TEXT PRIMARY KEY, path TEXT, row_idx INTEGER)")
    files = sorted(f for f in os.listdir(ENWIKI_DIR) if f.endswith(".parquet"))
    for i, f in enumerate(files):
        p = os.path.join(ENWIKI_DIR, f)
        names = pq.read_table(p, columns=["name"])["name"].to_pylist()
        conn.executemany(
            "INSERT OR REPLACE INTO wiki_loc VALUES (?,?,?)",
            [(n, p, idx) for idx, n in enumerate(names) if n],
        )
        if (i + 1) % 40 == 0 or i + 1 == len(files):
            conn.commit()
            print(f"  {i+1}/{len(files)} files  {time.time()-t0:.0f}s")
    conn.close()
    print(f"done -> {Path(LOC_DB).stat().st_size/1e6:.0f} MB in {time.time()-t0:.0f}s")

# shared read connection used by the retriever
_test_conn = sqlite3.connect(LOC_DB, check_same_thread=False)
print("_test_conn ready:", _test_conn.execute("SELECT COUNT(*) FROM wiki_loc").fetchone()[0], "titles")

Building title->location map ...
  40/265 files  11s
  80/265 files  24s
  120/265 files  39s
  160/265 files  53s
  200/265 files  68s
  240/265 files  83s
  265/265 files  94s
done -> 1529 MB in 94s
_test_conn ready: 7183892 titles


In [ ]:
# Shared cross-encoder loader. Used only by the use_reranker=True comparison paths
# (both Wikipedia and News); the routed retrievers feed full text and never call it.

_RERANKER_INSTANCE = None
def _get_reranker():
    """Load the cross-encoder lazily, reuse across retrievers."""
    global _RERANKER_INSTANCE
    if _RERANKER_INSTANCE is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        _RERANKER_INSTANCE = CrossEncoder(
            "cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=512, device=device,
        )
    return _RERANKER_INSTANCE


In [ ]:
# Offline Wikipedia article fetch — pull full text straight from the Kaggle dump.
# No network: the LLM names the title (next cell), we resolve it against wiki_loc
# and read the row from parquet, fresh each call (no cache). The parquet files have
# many row groups per file, so the global row index is mapped to (row_group, local)
# using the real row-group sizes — assuming one 25000-row group per file silently
# fetched empty text for canonical articles past the first group.

reranker = _get_reranker()        # shared cross-encoder (used by the reranked toggle)

def _flatten_sections(sections_json):
    try: secs = json.loads(sections_json) if isinstance(sections_json, str) else sections_json
    except Exception: return ""
    out = []
    def walk(node):
        if isinstance(node, dict):
            if node.get("type") == "paragraph" and node.get("value"):
                out.append(node["value"])
            for v in node.values(): walk(v)
        elif isinstance(node, list):
            for v in node: walk(v)
    walk(secs)
    return "\n".join(out)


_RG_OFFSETS = {}   # path -> (ParquetFile, [start row index of each row group])

def _rg_index(path):
    if path not in _RG_OFFSETS:
        pf = pq.ParquetFile(path)
        starts, base = [], 0
        for g in range(pf.num_row_groups):
            starts.append(base)
            base += pf.metadata.row_group(g).num_rows
        _RG_OFFSETS[path] = (pf, starts)
    return _RG_OFFSETS[path]


def fetch_article_offline(title, max_chars=8000):
    """Full article text from the Kaggle dump, read fresh from parquet (no cache)."""
    loc = _test_conn.execute(
        "SELECT path, row_idx FROM wiki_loc WHERE name=?", (title,)).fetchone()
    if loc is None:
        return ""
    path, idx = loc
    pf, starts = _rg_index(path)
    rg = bisect.bisect_right(starts, idx) - 1
    if rg < 0:
        return ""
    local = idx - starts[rg]
    tbl = pf.read_row_group(rg, columns=["name", "abstract", "sections"])
    names = tbl["name"].to_pylist()
    if local >= len(names) or names[local] != title:
        try: local = names.index(title)
        except ValueError: return ""
    abstract = tbl["abstract"].to_pylist()[local] or ""
    body = _flatten_sections(tbl["sections"].to_pylist()[local])
    return (abstract + "\n" + body).strip()[:max_chars]


def _is_disambig(text):
    head = text[:400].lower()
    return (len(text) < 250 or "may refer to" in head or "may also refer to" in head
            or "refers to" in head or "describes something" in head
            or "is the name of" in head)


def _hsent(text):
    """Sentence split with dedup (the dump repeats abstract sentences in the body)."""
    sents, seen = [], set()
    for s in re.split(r"(?<=[.!?])\s+(?=[A-Z])", text):
        s = s.strip()
        if len(s) < 40: continue
        k = s[:80].lower()
        if k in seen: continue
        seen.add(k); sents.append(s)
    return sents

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
# LLM-named titles -> offline resolution against wiki_loc.
# The LLM supplies the popularity/disambiguation signal the dump lacks (e.g. it
# names 'Augustus' or 'Pulp Fiction' directly), so we resolve fully offline.

def _loc_lookup(name):
    """Exact (PK, fast) then case-insensitive (indexed) title lookup in wiki_loc."""
    row = _test_conn.execute("SELECT name FROM wiki_loc WHERE name=?", (name,)).fetchone()
    if row: return row[0]
    row = _test_conn.execute("SELECT name FROM wiki_loc WHERE LOWER(name)=?",
                             (name.lower(),)).fetchone()
    return row[0] if row else None

_DISAMBIG_KINDS = "film|movie|character|TV series|song|album|band|novel|book|video game"

def resolve_offline_title(llm_title):
    """Map an LLM-named title to a real wiki_loc title, trying sensible variants
    (underscores/hyphens -> space, leading 'The', 'X film' -> 'X (film)')."""
    t = llm_title.strip().strip('"\'')
    base = re.sub(r"\s+", " ", t.replace("_", " ")).strip()
    variants = [base]
    if "-" in base:
        variants.append(re.sub(r"\s+", " ", base.replace("-", " ")).strip())
    extra = []
    for v in variants:
        if v.lower().startswith("the "):
            extra.append(v[4:])
        m = re.match(rf"(.+?)\s+\(?({_DISAMBIG_KINDS})\)?$", v, re.I)
        if m:
            extra.append(f"{m.group(1).strip()} ({m.group(2).lower()})")
    seen = set()
    for v in variants + extra:
        k = v.lower()
        if not v or k in seen: continue
        seen.add(k)
        hit = _loc_lookup(v)
        if hit:
            return hit
    return None

def llm_titles(question, options, max_titles=3):
    opts = "\n".join(f"- {o}" for o in (options or []))
    sys  = ("You name Wikipedia article titles. Given a trivia question and its options, "
            "list 1-3 exact English Wikipedia article titles whose content would best help "
            "answer it. One title per line. No numbering, no commentary.")
    user = f"Question: {question}\nOptions:\n{opts}\n\nWikipedia article titles:"
    raw, _, _ = _generate_with_messages(
        model, sys, user,
        {"max_new_tokens": 40, "do_sample": False, "return_dict_in_generate": True},
    )
    lines = [re.sub(r"^[\-\*\d\.\)\s]+", "", ln).strip()
             for ln in raw.splitlines() if ln.strip()]
    return raw, [ln for ln in lines if 2 <= len(ln) <= 80][:max_titles]

In [ ]:
# A single Wikipedia retriever for Entertainment (0), Ancient History (1) and Science (2).
#
#   llm_titles  ->  resolve_offline_title (wiki_loc)  ->  fetch_article_offline
#
# use_reranker=False (default, routed): feed the model the capped article text and
#   let it locate the answer. Beat every reranked variant -- the 14B model reading
#   the article outperforms a MiniLM cross-encoder picking sentences.
# use_reranker=True (comparison only, NOT routed): cross-encoder reranks the
#   article's sentences and feeds the top CHUNKS instead of the full text.

class WikipediaRetriever:
    def __init__(self, use_reranker=False, max_articles=2,
                 per_article_chars=8000, total_chars=16000,
                 chunk_context_chars=2500, use_options_in_rerank=True):
        self.use_reranker = use_reranker
        self.max_articles = max_articles
        self.per_article_chars = per_article_chars
        self.total_chars = total_chars
        self.chunk_context_chars = chunk_context_chars
        self.use_options_in_rerank = use_options_in_rerank
        # reranked mode needs deep text to rank over; full mode feeds what it fetches
        self.fetch_chars = max(per_article_chars, 16000) if use_reranker else per_article_chars

    def _gather(self, question, options=None):
        _, titles = llm_titles(question, options)
        out, seen = [], set()
        for t in titles:
            name = resolve_offline_title(t)
            if not name or name.lower() in seen:
                continue
            text = fetch_article_offline(name, max_chars=self.fetch_chars)
            if text and not _is_disambig(text):
                out.append((name, text)); seen.add(name.lower())
            if len(out) >= self.max_articles:
                break
        return out

    # ---- full-article mode (default keeper) ----
    def _format_full(self, arts):
        out, total = [], 0
        for title, text in arts:
            budget = min(self.per_article_chars, self.total_chars - total)
            if budget <= 0:
                break
            out.append(f"== {title} ==\n{text[:budget]}")
            total += min(len(text), budget)
        return "\n\n".join(out)

    # ---- reranked-chunk mode (comparison only) ----
    def _rank(self, question, options, arts):
        passages, seen = [], set()
        for title, text in arts:
            for s in _hsent(text):
                k = s[:80].lower()
                if k in seen: continue
                seen.add(k); passages.append((title, s))
        if not passages:
            return []
        texts = [p[1] for p in passages]
        score = np.array(reranker.predict([(question, t) for t in texts]), dtype=float)
        if self.use_options_in_rerank and options:
            opt_mat = [reranker.predict([(f"{question} {opt}", t) for t in texts])
                       for opt in options]
            score = np.maximum(score, np.max(np.array(opt_mat, dtype=float), axis=0))
        return sorted(zip(score.tolist(), passages), key=lambda x: x[0], reverse=True)

    def _format_chunks(self, ranked):
        out, total = [], 0
        for score, (title, sent) in ranked:
            if total + len(sent) > self.chunk_context_chars and out:
                break
            out.append(f"[{title}] {sent}")
            total += len(sent)
        return "\n".join(out)

    def retrieve_and_format(self, question, options=None, competition_id=None):
        arts = self._gather(question, options)
        if not arts:
            return ""
        if self.use_reranker:
            return self._format_chunks(self._rank(question, options, arts))
        return self._format_full(arts)

    def retrieve_debug(self, question, options=None):
        arts = self._gather(question, options)
        if self.use_reranker:
            ranked = self._rank(question, options, arts)
            return {"articles": [t for t, _ in arts],
                    "top": [(float(s), p[1][:110]) for s, p in ranked[:6]]}
        return {"articles": [t for t, _ in arts],
                "top": [(0.0, txt[:110]) for _, txt in arts]}

In [ ]:
# Google News RSS source + article-body fetching.
#
# Google News snippets are short; the answer is often only in the article body, so
# we decode the Google redirect URL to the real publisher, extract clean text with
# trafilatura, and fall back to the Jina Reader API. Bodies augment the snippet.

class GoogleNewsSource:
    """Google News RSS — recent news from major outlets. Free, no API key.
    Returns the article URL so WithArticleBodies can fetch the full body."""
    RSS_URL = "https://news.google.com/rss/search"
    HEADERS = {"User-Agent": "PoliMillionaire-NLP-Project/1.0 (educational; PoliMi student project)"}

    def __init__(self, max_chars=1500, timeout=5.0):
        self.max_chars = max_chars
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update(self.HEADERS)

    def search(self, query, k=3, mode="text", **kwargs):
        params = {"q": query, "hl": "en-US", "gl": "US", "ceid": "US:en"}
        results = []
        try:
            r = self.session.get(self.RSS_URL, params=params, timeout=self.timeout)
            r.raise_for_status()
            root = ET.fromstring(r.content)
            for item in root.findall(".//item")[:k]:
                title = (item.findtext("title", "") or "").strip()
                desc = (item.findtext("description", "") or "").strip()
                desc = re.sub(r"<[^>]+>", " ", desc)
                desc = re.sub(r"\s+", " ", html.unescape(desc)).strip()
                source_elem = item.find("source")
                publisher = (source_elem.text or "") if source_elem is not None else ""
                parts = [title]
                if publisher and publisher.lower() not in title.lower():
                    parts.append(f"({publisher})")
                if desc and desc.lower() not in title.lower():
                    parts.append(desc)
                text = " \u2014 ".join(parts)[: self.max_chars]
                if text:
                    results.append({"title": title[:120], "text": text,
                                    "url": item.findtext("link", "") or "", "source": "google-news"})
        except Exception:
            results = []
        return results


_PAYWALL_DOMAINS = {
    "nytimes.com", "wsj.com", "ft.com", "bloomberg.com", "economist.com",
    "newyorker.com", "thetimes.co.uk", "telegraph.co.uk", "wired.com",
}

def _decode_google_url(url):
    """Unwrap Google News redirect URLs to the real publisher URL."""
    if not url or "news.google.com" not in url:
        return url
    try:
        result = gnewsdecoder(url)
        if result.get("status"):
            return result.get("decoded_url", url)
    except Exception:
        pass
    return url

def _fetch_url_text(url, timeout=3.0, max_chars=3000):
    """Fetch article body: trafilatura primary, Jina Reader fallback.
    Skips known-paywall domains. Returns empty string on failure."""
    if not url:
        return ""
    real_url = _decode_google_url(url)
    if any(d in real_url for d in _PAYWALL_DOMAINS):
        return ""
    try:
        r = requests.get(real_url, timeout=timeout, allow_redirects=True,
                         headers={"User-Agent": "Mozilla/5.0 (compatible; PoliMillionaire-NLP/1.0)"})
        if r.status_code == 200 and r.content:
            text = trafilatura.extract(r.text, include_links=False, include_images=False,
                                       include_tables=False, include_comments=False)
            if text and len(text) >= 100:
                return text[:max_chars]
    except Exception:
        pass
    try:
        r = requests.get(f"https://r.jina.ai/{real_url}", timeout=timeout + 3,
                         headers={"Accept": "text/markdown"})
        if r.status_code == 200 and len(r.text) >= 100:
            return r.text[:max_chars]
    except Exception:
        pass
    return ""


class WithArticleBodies:
    """Wrap a source; enrich top-N results with the full article body (prepended to
    the snippet so verbatim headline matches survive)."""
    def __init__(self, base_source, fetch_top=3, body_timeout=3.0,
                 body_max_chars=2500, max_workers=3):
        self.base = base_source
        self.fetch_top = fetch_top
        self.body_timeout = body_timeout
        self.body_max_chars = body_max_chars
        self.max_workers = max_workers

    def search(self, query, k=3, mode="text", **kwargs):
        results = self.base.search(query, k=k, mode=mode)
        if not results:
            return results
        bodies = {}
        to_fetch = [(i, r.get("url", "")) for i, r in enumerate(results[: self.fetch_top])
                    if r.get("url", "")]
        if to_fetch:
            with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as ex:
                fut = {ex.submit(_fetch_url_text, url, self.body_timeout, self.body_max_chars): url
                       for _, url in to_fetch}
                try:
                    for future in concurrent.futures.as_completed(fut, timeout=self.body_timeout + 2):
                        url = fut[future]
                        try: bodies[url] = future.result()
                        except Exception: bodies[url] = ""
                except concurrent.futures.TimeoutError:
                    for url in fut.values():
                        bodies.setdefault(url, "")
        enriched = []
        for i, r in enumerate(results):
            body = bodies.get(r.get("url", ""), "")
            if body and i < self.fetch_top:
                r = dict(r)
                r["text"] = (r["text"] + "\n\n" + body)[: self.body_max_chars + 500]
                r["source"] = r["source"] + "+body"
            enriched.append(r)
        return enriched

In [ ]:
# A single News retriever for the News category (5), mirroring WikipediaRetriever.
#
#   GoogleNewsSource (RSS)  ->  WithArticleBodies (decode + trafilatura/Jina)  ->  LLM
#
# use_reranker=False (default, routed): feed the model the top article BODIES
#   concatenated (capped) and let it read across recent articles. The keeper.
# use_reranker=True (comparison only, NOT routed): cross-encoder reranks sentences
#   from the bodies and feeds the top CHUNKS instead of the full bodies.

class NewsRetriever:
    def __init__(self, use_reranker=False, fetch_top=3, body_timeout=3.0,
                 per_article_chars=2500, total_chars=6000, chunk_context_chars=3000,
                 use_options_in_rerank=False):
        self.use_reranker = use_reranker
        self.fetch_top = fetch_top
        self.per_article_chars = per_article_chars
        self.total_chars = total_chars
        self.chunk_context_chars = chunk_context_chars
        self.use_options_in_rerank = use_options_in_rerank
        self.source = WithArticleBodies(GoogleNewsSource(), fetch_top=fetch_top,
                                        body_timeout=body_timeout)

    def _gather(self, question):
        return [r for r in self.source.search(question, k=self.fetch_top, mode="text")
                if (r.get("text") or "").strip()]

    # ---- full-bodies mode (default keeper) ----
    def _format_full(self, results):
        out, total = [], 0
        for r in results[: self.fetch_top]:
            text = (r.get("text") or "").strip()
            budget = min(self.per_article_chars, self.total_chars - total)
            if budget <= 0:
                break
            title = r.get("title", r.get("source", "article"))
            out.append(f"== {title} ==\n{text[:budget]}")
            total += min(len(text), budget)
        return "\n\n".join(out)

    # ---- reranked-chunk mode (comparison only) ----
    def _rank(self, question, options, results):
        passages, seen = [], set()
        for r in results:
            title = r.get("title", r.get("source", "article"))
            for s in _hsent(r.get("text") or ""):
                k = s[:80].lower()
                if k in seen: continue
                seen.add(k); passages.append((title, s))
        if not passages:
            return []
        texts = [pp[1] for pp in passages]
        score = np.array(reranker.predict([(question, t) for t in texts]), dtype=float)
        if self.use_options_in_rerank and options:
            opt_mat = [reranker.predict([(f"{question} {opt}", t) for t in texts])
                       for opt in options]
            score = np.maximum(score, np.max(np.array(opt_mat, dtype=float), axis=0))
        return sorted(zip(score.tolist(), passages), key=lambda x: x[0], reverse=True)

    def _format_chunks(self, ranked):
        out, total = [], 0
        for score, (title, sent) in ranked:
            if total + len(sent) > self.chunk_context_chars and out:
                break
            out.append(f"[{title}] {sent}")
            total += len(sent)
        return "\n".join(out)

    def retrieve_and_format(self, question, options=None, competition_id=None):
        results = self._gather(question)
        if not results:
            return ""
        if self.use_reranker:
            return self._format_chunks(self._rank(question, options, results))
        return self._format_full(results)

    def retrieve_debug(self, question, options=None):
        results = self._gather(question)
        if self.use_reranker:
            ranked = self._rank(question, options, results)
            return {"articles": [r.get("title", "?") for r in results],
                    "top": [(float(s), pp[1][:110]) for s, pp in ranked[:6]]}
        return {"articles": [r.get("title", "?") for r in results],
                "top": [(0.0, (r.get("text") or "")[:110]) for r in results]}

In [ ]:
# Per-competition routing. Dispatch to a retriever by competition id, or disable.

class RoutingRetriever:
    def __init__(self, routes, default=None, disable_for=None):
        self.routes = routes
        self.default = default
        self.disable_for = disable_for or set()

    def retrieve_and_format(self, question, options=None, competition_id=None):
        if competition_id in self.disable_for:
            return ""
        ret = self.routes.get(competition_id, self.default)
        if ret is None:
            return ""
        return ret.retrieve_and_format(question, options, competition_id)


# Wikipedia (Entertainment + Ancient History), fully offline + LLM-named titles
wiki_full     = WikipediaRetriever(use_reranker=False)   # the keeper -- routed
wiki_reranked = WikipediaRetriever(use_reranker=True)    # comparison only -- NOT routed

# News, full article bodies fed to the LLM
news_full     = NewsRetriever(use_reranker=False)        # the keeper -- routed
news_reranked = NewsRetriever(use_reranker=True)         # comparison only -- NOT routed

# Routing:  0,1,2 -> Wikipedia (full article) ;  5 -> News (full bodies) ;  3,4 -> no RAG
routing_retriever = RoutingRetriever(
    routes={0: wiki_full, 1: wiki_full, 2: wiki_full, 5: news_full},
    default=None,
    disable_for={3, 4},
)
retriever = routing_retriever        # used by ask_llm when use_rag=True

# Pipeline & Wrapper

In [ ]:
# whole message-to-llm builder

def build_messages(model_name, system_prompt, user_prompt):
    model_name = model_name.lower()

    # GEMMA 3 FORMAT
    if is_gemma3(model_name):
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt,}]},
            {"role": "user", "content": [{"type": "text", "text": user_prompt,}]}
        ]
        return messages

    # GEMMA 2 FORMAT
    elif is_gemma2(model_name):
        merged_prompt = (
            system_prompt.strip()
            + "\n\n"
            + user_prompt.strip()
        )
        messages = [
            {"role": "user", "content": merged_prompt,}
        ]
        return messages

    # STANDARD CHAT TEMPLATE MODELS
    else:
        messages = [
            {"role": "system", "content": system_prompt,},
            {"role": "user", "content": user_prompt,},
        ]
        return messages

In [ ]:
# clean model output from text and logits

def extract_answer_from_text(text):
    """
    Parse the model's generated string for a 1-4 answer.
    Returns int in {1,2,3,4} or None if no digit found.
    """
    text = text.strip()
    match = re.search(r"[1-4]", text)
    return int(match.group()) if match else None


def extract_answer_from_logits(scores_step0, decode_fn, top_k=50):
    """
    Compute answer probabilities by inspecting the model's first-step logits.
    Walks the top-k most likely tokens at the first generation position and
    aggregates probability mass over those that decode to '1', '2', '3', '4'
    (stripped). This handles tokenizer quirks (leading-space tokens etc.)
    without needing per-tokenizer special cases.
    Returns:
        best_answer: int in {1,2,3,4} or None if no answer-token in top-k
        probs: dict {1: p1, 2: p2, 3: p3, 4: p4}, renormalized to sum to 1
    """
    probs = torch.softmax(scores_step0[0].float(), dim=-1)
    top_probs, top_ids = probs.topk(top_k)

    answer_probs = {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0}
    for i in range(top_k):
        tok_id = top_ids[i].item()
        try:
            decoded = decode_fn(tok_id).strip()
        except Exception:
            continue
        if decoded in ("1", "2", "3", "4"):
            answer_probs[int(decoded)] += top_probs[i].item()

    total = sum(answer_probs.values())
    if total == 0:
        # No answer token found in the top-k; fall back to uniform
        return None, {k: 0.25 for k in (1, 2, 3, 4)}

    answer_probs = {k: v / total for k, v in answer_probs.items()}
    best = max(answer_probs, key=answer_probs.get)
    return best, answer_probs

In [ ]:
# answer functions

def _decode_fn_for_model():
    """
    Return a unified decode_fn(tok_id) -> str for the active model family.
    """
    if is_mistral3(MODEL_NAME):
        return lambda tid: mistral_tokenizer.decode([tid])
    elif is_gemma3(MODEL_NAME):
        return lambda tid: processor.tokenizer.decode([tid])
    else:
        return lambda tid: tokenizer.decode([tid])


def _generate_with_messages(model, system_prompt, user_prompt, gen_config):
    """
    One forward pass. Returns (raw_output, latency, first-step logits).
    """
    messages = build_messages(MODEL_NAME, system_prompt, user_prompt)

    if is_mistral3(MODEL_NAME):
        tokenized = mistral_tokenizer.encode_chat_completion(
            ChatCompletionRequest(messages=messages)
        )
        input_ids = torch.tensor([tokenized.tokens], device=model.device)
        attention_mask = torch.ones_like(input_ids)
        gen_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "pad_token_id": 2,
            **gen_config,
        }
        input_len = len(tokenized.tokens)
        decode_seq = lambda toks: mistral_tokenizer.decode(toks.tolist()).strip()

    elif is_gemma3(MODEL_NAME):
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt",
        ).to(model.device, dtype=torch.bfloat16)
        gen_kwargs = {**inputs, **gen_config}
        input_len = inputs["input_ids"].shape[-1]
        decode_seq = lambda toks: processor.decode(toks, skip_special_tokens=True).strip()

    else:
        prompt_str = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)
        gen_kwargs = {**inputs, "pad_token_id": tokenizer.eos_token_id, **gen_config}
        input_len = inputs["input_ids"].shape[1]
        decode_seq = lambda toks: tokenizer.decode(toks, skip_special_tokens=True).strip()

    start = time.time()
    with torch.inference_mode():
        gen_out = model.generate(**gen_kwargs)
    latency = time.time() - start

    sequences = gen_out.sequences
    generated_tokens = sequences[0][input_len:]
    raw_output = decode_seq(generated_tokens)
    scores_step0 = gen_out.scores[0] if gen_out.scores else None
    return raw_output, latency, scores_step0


def ask_llm(model, question, option_texts, competition_id=None,
            strategy=None, use_rag=None):
    """
    Run inference with a prompt strategy.
    Returns dict:
        answer       : int 1-4 or None
        confidence   : dict {1..4: prob}
        raw_output   : str (single) or list[str] (multi-sample)
        latency      : total seconds across all samples (generation only)
        rag_latency  : seconds spent in retrieval (0 if RAG off)
        samples      : list of per-sample answers (None if n_samples=1)

    Args:
        use_rag: explicit override. If None, defaults to strategy.use_rag.
    """
    if strategy is None:
        strategy = P0_BASELINE
    if use_rag is None:
        use_rag = strategy.use_rag

    system_prompt = strategy.get_system(competition_id)
    user_prompt = strategy.user_builder(question, option_texts, competition_id)

    # --- RAG: retrieve and prepend context to the user prompt ------------
    rag_latency = 0.0
    if use_rag:
        rag_start = time.time()
        context = retriever.retrieve_and_format(question, option_texts, competition_id)
        rag_latency = time.time() - rag_start
        if context:
            user_prompt = (
                "Use the following reference information from external sources to help "
                "you answer the question. If the references are not relevant, "
                "rely on your own knowledge.\n\n"
                f"References:\n{context}\n\n"
                "---\n\n"
                f"{user_prompt}"
            )
    # ---------------------------------------------------------------------

    gen_config = {
        "use_cache": True,
        "output_scores": True,
        "return_dict_in_generate": True,
        "max_new_tokens": strategy.max_new_tokens,
        "do_sample": strategy.do_sample,
    }
    if strategy.do_sample:
        gen_config["temperature"] = strategy.temperature

    decode_fn = _decode_fn_for_model()
    answers, raw_outputs, confidences = [], [], []
    total_latency = 0.0

    for _ in range(strategy.n_samples):
        raw_output, latency, scores_step0 = _generate_with_messages(
            model, system_prompt, user_prompt, gen_config
        )
        raw_outputs.append(raw_output)
        total_latency += latency
        _, conf = extract_answer_from_logits(scores_step0, decode_fn) \
            if scores_step0 is not None else (None, {k: 0.25 for k in (1, 2, 3, 4)})
        confidences.append(conf)
        answers.append(strategy.parser(raw_output, conf))

    if strategy.n_samples > 1:
        valid = [a for a in answers if a is not None]
        if valid:
            counts = Counter(valid)
            final_answer = counts.most_common(1)[0][0]
            total = sum(counts.values())
            final_confidence = {k: counts.get(k, 0) / total for k in (1, 2, 3, 4)}
        else:
            final_answer = None
            final_confidence = {k: 0.25 for k in (1, 2, 3, 4)}
        return {
            "answer": final_answer,
            "confidence": final_confidence,
            "raw_output": raw_outputs,
            "latency": total_latency,
            "rag_latency": rag_latency,
            "samples": answers,
        }

    return {
        "answer": answers[0],
        "confidence": confidences[0],
        "raw_output": raw_outputs[0],
        "latency": total_latency,
        "rag_latency": rag_latency,
        "samples": None,
    }

In [ ]:
# sanity test
question = "What is the capital of Australia?"
option_texts = ["Sydney", "Melbourne", "Canberra", "Perth"]

result = ask_llm(model, question, option_texts, use_rag=True)

print("Question:", question)
print("Options:", option_texts)
print("Raw output     :", repr(result["raw_output"]))
print("Answer         :", result["answer"])
print("Confidence     :", {k: f"{v:.3f}" for k, v in result["confidence"].items()})
print("Latency        : {:.3f}s".format(result["latency"]))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: What is the capital of Australia?
Options: ['Sydney', 'Melbourne', 'Canberra', 'Perth']
Raw output     : '3'
Answer         : 3
Confidence     : {1: '0.000', 2: '0.000', 3: '1.000', 4: '0.000'}
Latency        : 3.269s


In [ ]:
# --- smoke test ---
print("=== Routing retriever sanity test ===\n")
for label, comp_id, q, opts in [
    ("Entertainment", 0, "Who directed the film Pulp Fiction?",
     ["Martin Scorsese", "Quentin Tarantino", "Steven Spielberg", "Ridley Scott"]),
    ("AncientHist",   1, "Who was the first Roman emperor?",
     ["Julius Caesar", "Augustus", "Nero", "Tiberius"]),
    ("Science(off)",  2, "What gas do plants absorb for photosynthesis?",
     ["Oxygen", "Carbon dioxide", "Nitrogen", "Hydrogen"]),
    ("News",          5, "Who won the 2024 US presidential election?",
     ["Joe Biden", "Donald Trump", "Kamala Harris", "Nikki Haley"]),
]:
    ctx = routing_retriever.retrieve_and_format(q, opts, competition_id=comp_id)
    print(f"{label:14} (comp {comp_id}) -> {len(ctx):4d} chars; {ctx[:110]!r}\n")

=== Routing retriever sanity test ===

Entertainment  (comp 0) -> 16048 chars; '== Quentin Tarantino ==\nQuentin Jerome Tarantino is an American filmmaker, actor, and author. His films are ch'

AncientHist    (comp 1) -> 16037 chars; '== Augustus ==\nAugustus, also known as Octavian, was the founder of the Roman Empire and the first Roman emper'

Science(off)   (comp 2) -> 16046 chars; '== Carbon dioxide ==\nCarbon dioxide is a chemical compound with the chemical formula CO2. It is made up of mol'

News           (comp 5) -> 6337 chars; '== US presidential election results 2024: Harris vs. Trump | Live maps, charts and the latest vote count - Reu'



# Polimillionaire Gameplay

In [ ]:
# api retry helpers

def _is_transient_api_error(e):
    """Heuristic: connection errors that may succeed on retry."""
    err_str = str(e).lower()
    return any(token in err_str for token in [
        "connection", "remotedisconnected", "timed out",
        "aborted", "broken pipe", "reset by peer",
    ])


def _retry_api(fn, max_retries=3, base_wait=1.0, label="API call"):
    """Call fn(); on transient errors, exponential-backoff retry.

    Returns the result. Re-raises the last exception if all retries fail
    or the error isn't transient.
    """
    last_exc = None
    for attempt in range(max_retries):
        try:
            return fn()
        except Exception as e:
            last_exc = e
            if not _is_transient_api_error(e):
                raise
            if attempt < max_retries - 1:
                wait = base_wait * (2 ** attempt)
                print(f"  ⚠ {label} failed (attempt {attempt + 1}/{max_retries}): {e}")
                print(f"    Retrying in {wait:.1f}s...")
                time.sleep(wait)
    raise last_exc

In [ ]:
def play_leaderboard(model, game, eval_log=None, verbose=True, strategy=None, use_rag=None):
    """
    Play one game against the live API. Robust to mid-game API failures:
    retries transient connection errors; on permanent failure, logs an
    'api_error' row and ends the game gracefully (no exception raised).
    """
    if strategy is None:
        strategy = P0_BASELINE

    while game.in_progress:
        question = game.current_question
        if not question:
            if verbose:
                print("No question available. Game may have ended.")
            break

        option_texts = [opt.text for opt in question.options]
        comp_id = game.state.competition.id
        level = game.current_level

        if verbose:
            print(f"\n--- Level {level} ---")
            print(f"Q: {question.text}")
            for i, opt in enumerate(question.options):
                print(f"  [{i+1}] {opt.text}")
            tl = game.time_remaining
            if tl:
                print(f"Time remaining: {tl:.1f}s")

        if comp_id == 3:
            # Maths: chain-of-thought (no RAG, no tools) -- best on comp 3
            result = ask_llm(model, question.text, option_texts,
                             competition_id=comp_id, strategy=P_MATHS_COT, use_rag=False)
        else:
            result = ask_llm(model, question.text, option_texts,
                             competition_id=comp_id, strategy=strategy, use_rag=use_rag)
        answer_int = result["answer"]
        if answer_int is None:
            answer_int = max(result["confidence"], key=result["confidence"].get)
            if verbose:
                print(f"Parser failed on {result['raw_output']!r}; "
                      f"fell back to confidence-argmax = {answer_int}")

        chosen_idx = answer_int - 1
        chosen_option_id = question.options[chosen_idx].id

        if verbose:
            print(f"Chose option {answer_int} -> '{option_texts[chosen_idx]}' "
                  f"(latency {result['latency']:.2f}s, "
                  f"conf {result['confidence'][answer_int]:.2f})")

        # Submit answer with retry on transient errors
        api_result = None
        api_error_msg = None
        try:
            api_result = _retry_api(
                lambda: game.answer(chosen_option_id),
                max_retries=3,
                label=f"game.answer (Comp {comp_id} L{level})",
            )
        except Exception as e:
            api_error_msg = str(e)
            if verbose:
                print(f"  ✗ Permanent API failure at L{level}: {e}")

        total_lat = result["latency"] + result.get("rag_latency", 0.0)

        if eval_log is not None:
            eval_log.append({
                "strategy": strategy.name,
                "competition": comp_id,
                "level": level,
                "question": question.text,
                "options": option_texts,
                "predicted_idx": chosen_idx,
                "answer": result["answer"],
                "confidence": result["confidence"],
                "raw_output": result["raw_output"],
                "latency": result["latency"],
                "rag_latency": result.get("rag_latency", 0.0),
                "total_latency": total_lat,
                "timeout_risk": total_lat > LATENCY_DANGER_THRESHOLD_S,
                "correct": bool(api_result.correct) if api_result else False,
                "timed_out": bool(api_result.timed_out) if api_result else False,
                "earned_after": api_result.earned_amount if api_result else 0.0,
                "api_error": api_error_msg,
            })

        if api_result is None:
            # Permanent failure: end this game gracefully
            break

        if api_result.timed_out:
            if verbose:
                print(f"TIMED OUT at level {level}. Final: ${api_result.earned_amount:,.2f}")
            break
        if not api_result.correct:
            if verbose:
                print(f"WRONG. Final: ${api_result.earned_amount:,.2f}")
            break
        if verbose:
            print(f"CORRECT. Earned so far: ${api_result.earned_amount:,.2f}")

    if verbose:
        print(f"\n=== Game over === Reached level {game.current_level}, "
              f"earned ${game.earned_amount:,.2f}")
    return game

In [ ]:
# show leaderboard position
#lb = client.leaderboard.get(competition_id=5, limit=50)
#print(f"\n=== Leaderboard for {lb.competition.name} ===")
#for i, entry in enumerate(lb.entries, 1):
#    marker = " <-- YOU" if entry.username == username else ""
#    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")

# Speech Mode

Whisper transcription + an LLM transcript-cleaning pass, plus a unified
game loop (`play_leaderboard_unified`) that handles both text and speech games.


In [ ]:
!pip -q install faster-whisper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 56.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 53.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.2 MB/s eta 0:00:00:00:0100:01


In [ ]:
def parse_full_text(raw_output, confidence):
    return raw_output.strip().split('\n')[0].strip()


def clean_prompt_builder(raw_text, options, comp_id=None):
    context_q = options[0] if options else None

    prompt = ""
    # add the question to the option context so the LLM can reinterpret misspelled words more easily
    if context_q:
        prompt += f"Question Context: {context_q}\n"

    prompt += f"Raw Audio: {raw_text}\n"
    prompt += "Clean Text:"
    return prompt

# --- STRATEGY 1: FOR THE OPTIONS ---
SPEECH_CLEAN_OPTION_STRATEGY = PromptStrategy(
    name="Transcript_Cleaner_Options",
    system_prompt=(
        "You are a strict transcript cleaner, trivia expert, and phonetic decoder for a quiz game.\n"
        "You receive raw, noisy speech-to-text transcripts of answer options containing fillers (um, uh, ahem), laughter, stutters, and severe phonetic misinterpretations.\n\n"
        "Instructions:\n"
        "1. DYNAMIC TRIVIA RE-INTERPRETATION: Read the 'Question Context' first. Examine the 'Raw Audio' transcript for misspelled, mangled, or nonsensical words. Ask yourself: 'What actual trivia option/proper noun is the user trying to say that fits this question?' Reverse-engineer the Whisper transcriber's phonetic mistakes to output a highly compatible, real-world trivia option.\n"
        "2. STRIP PREFIXES, LETTERS & LAUGHTER: Completely remove option announcements, structural prefixes, and vocal noise. Permanently delete letters or words like 'Option A.', 'Option B!', 'B.', 'C,', 'ha ha', 'he he', 'wow'. Do not include them in the clean text.\n"
        "3. STRIP TRAILING PUNCTUATION: Remove all trailing periods, commas, or exclamation points.\n"
        "4. STRICT OUTPUT: Return ONLY the final corrected option text on a single line. No markdown formatting, no conversational framing, and no explanation.\n\n"
        "Examples:\n"
        "Question Context: What term describes the earliest form of the Greek language?\n"
        "Raw Audio: Option B! Ha ha! Add a Greek!\n"
        "Clean Text: Attic Greek\n\n"
        "Question Context: What term describes the earliest form of the Greek language?\n"
        "Raw Audio: Option C. He ha ha ha ha ha ha ha ha ha ha ha ha ha ancient Greek.\n"
        "Clean Text: Ancient Greek\n\n"
        "Question Context: Which country won the 2018 FIFA World Cup?\n"
        "Raw Audio: Option D... um, Friends!\n"
        "Clean Text: France"
    ),
    user_builder=clean_prompt_builder,
    parser=parse_full_text,
    max_new_tokens=32, #short to truncate possible allucinations
    do_sample=False,
    n_samples=1
)

# --- STRATEGY 2: FOR THE MAIN QUESTION ---
SPEECH_CLEAN_QUESTION_STRATEGY = PromptStrategy(
    name="Transcript_Cleaner_Question",
    system_prompt=(
        "You are a strict transcript cleaner for a trivia game. "
        "You receive the raw speech-to-text transcript of the MAIN QUIZ QUESTION.\n\n"
        "Instructions:\n"
        "1. DO NOT ANSWER THE QUESTION. You are a transcriber, not a player. Your only job is to clean the text.\n"
        "2. KEEP ALL INFORMATION. Retain the full length, specific details, and original phrasing of the question.\n"
        "3. REMOVE NOISE: Strip out conversational fillers (um, uh, ahem, wow), false starts, and exaggerated exclamation marks.\n"
        "4. FIX PHONETICS: Correct obvious phonetic STT misinterpretations of historical or scientific terms.\n"
        "5. STRICT OUTPUT: Return EXACTLY the full, cleaned question on a single line. Do not add quotes, labels, or reasoning."
    ),
    user_builder=clean_prompt_builder,
    parser=parse_full_text,
    max_new_tokens=256,
    do_sample=False,
    n_samples=1
)

def clean_transcript(raw_text: str, label: str = "text", context_question: str = None) -> str:
    """Strip fillers using the recycled high-level ask_llm pattern, dynamically routing by label."""

    sneak_context = [context_question] if context_question else []

    if "question" in label.lower():
        active_strategy = SPEECH_CLEAN_QUESTION_STRATEGY
    else:
        active_strategy = SPEECH_CLEAN_OPTION_STRATEGY

    result = ask_llm(
        model=model,
        question=raw_text,
        option_texts=sneak_context,
        competition_id=label,
        strategy=active_strategy,
        use_rag=False
    )
    cleaned_text = active_strategy.parser(result["raw_output"], result.get("confidence"))

    return str(cleaned_text).strip()

# ── Mock objects ──────────────────────────────────────────────────────────────
from dataclasses import dataclass

@dataclass
class MockOption:
    id:   int
    text: str

@dataclass
class MockQuestion:
    text:    str
    options: list[MockOption]


def build_mock_question(
    question_audio: bytes,
    option_audios:  list[bytes],
    option_map:     dict[str, int],
    language:       str | None = None,
    verbose:        bool = True,
) -> MockQuestion:
    # ── question ──────────────────────────────────────────────────────────
    raw_q = transcribe_audio(question_audio, language=language)
    if verbose:
        print(f"  [question] RAW  : {raw_q}")
    clean_q = clean_transcript(raw_q, label="question")
    if verbose:
        print(f"  [question] CLEAN: {clean_q}")

    # ── options ───────────────────────────────────────────────────────────
    mock_options = []
    for i, opt_audio in enumerate(option_audios):
        letter     = chr(65 + i)
        server_id  = option_map.get(letter, i)

        raw_o = transcribe_audio(opt_audio, language=language)
        if verbose:
            print(f"  [option_{letter}] RAW  : {raw_o}")

        clean_o = clean_transcript(raw_o, label=f"option_{letter}", context_question=clean_q)

        if verbose:
            print(f"  [option_{letter}] CLEAN: {clean_o}")

        mock_options.append(MockOption(id=server_id, text=clean_o))

    return MockQuestion(text=clean_q, options=mock_options)

In [ ]:
import io
import torch
from faster_whisper import WhisperModel

WHISPER_MODEL_SIZE = "distil-large-v3"

SPEECH_LANGUAGE: str | None = "en"

_whisper_model = None

def _get_whisper() -> WhisperModel:
    global _whisper_model
    if _whisper_model is None:
        print(f"[speech] Loading Faster-Whisper '{WHISPER_MODEL_SIZE}' …")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        compute_type = "int8_float16" if device == "cuda" else "int8"

        _whisper_model = WhisperModel(
            WHISPER_MODEL_SIZE,
            device=device,
            compute_type=compute_type
        )
    return _whisper_model

def transcribe_audio(audio_bytes: bytes, language: str | None = None) -> str:
    """
    Transcribe raw WAV bytes with faster-whisper entirely in memory.
    """
    wmodel = _get_whisper()
    lang = language if language else SPEECH_LANGUAGE
    audio_stream = io.BytesIO(audio_bytes)

    try:
        segments, info = wmodel.transcribe(
            audio_stream,
            language=lang,
            beam_size=5
        )

        text = " ".join([segment.text for segment in segments])
        return text.strip()

    except Exception as e:
        print(f"[ERROR] Transcription failed: {e}")
        return ""

In [ ]:
def play_leaderboard_unified(model, game, strategy=None, use_rag=None,
                      question_bank=None, eval_log=None,
                      language="en", verbose=True):
    """
    A single, unified game loop handling both Text and Speech modes.
    """
    if strategy is None:
        strategy = P0_BASELINE

    is_speech_mode = getattr(game, "mode", "text") == "speech"

    if verbose:
        print(f"\n[SYSTEM] Starting game in {getattr(game, 'mode', 'text').upper()} mode.")
        print(f"[SYSTEM] Strategy: {strategy.name} | RAG: {use_rag if use_rag is not None else strategy.use_rag}")

    while game.in_progress:

        if is_speech_mode:
            if verbose: print(f"\n--- Level {game.current_level} (Speech Capture) ---")
            try:
                question_audio = game.fetch_audio_question()
                option_map, option_audios = {}, []

                for i in range(4):
                    letter = chr(65 + i)
                    opt_audio = game.fetch_audio_option_next()
                    option_audios.append(opt_audio)
                    if game.current_question and i < len(game.current_question.options):
                        option_map[letter] = game.current_question.options[i].id

                question = build_mock_question(
                    question_audio, option_audios, option_map,
                    language=language, verbose=verbose
                )
            except Exception as e:
                print(f"[ERROR] Audio acquisition failed: {e}"); break
        else:
            question = game.current_question
            if not question:
                if verbose: print("No question available. Game may have ended.")
                break
            if verbose:
                print(f"\n--- Level {game.current_level} (Text Mode) ---")
                print(f"Q: {question.text}")
                for i, opt in enumerate(question.options):
                    print(f"  [{i+1}] {opt.text}")

        #--------------------------------------------------------------
        #FROM HERE IT SHOULD BE THE SAME AS PREVIOUS IMPLEMENTATION !!!
        #--------------------------------------------------------------

        option_texts = [opt.text for opt in question.options]
        comp_id = game.state.competition.id
        level = game.current_level

        if is_speech_mode:
            game.refresh_state()

        tl = game.time_remaining
        if tl and verbose:
            print(f"Time remaining: {tl:.1f}s")

        result = ask_llm(model, question.text, option_texts,
                         competition_id=comp_id, strategy=strategy, use_rag=use_rag)

        answer_int = result["answer"]
        if answer_int is None:
            answer_int = max(result["confidence"], key=result["confidence"].get)
            if verbose: print(f"Parser fallback to logit-argmax = {answer_int}")

        chosen_idx = answer_int - 1
        chosen_option_id = question.options[chosen_idx].id

        if verbose:
            print(f"Chose option {answer_int} -> '{option_texts[chosen_idx]}' "
                  f"(latency {result['latency']:.2f}s, conf {result['confidence'][answer_int]:.2f})")

        api_result = game.answer(chosen_option_id)

        if eval_log is not None:
            eval_log.append({
                "mode": getattr(game, "mode", "text"),
                "strategy": strategy.name,
                "competition": comp_id,
                "level": level,
                "question": question.text,
                "options": option_texts,
                "predicted_idx": chosen_idx,
                "answer": result["answer"],
                "confidence": result["confidence"],
                "latency": result["latency"],
                "timeout_risk": result["latency"] > LATENCY_DANGER_THRESHOLD_S,
                "correct": bool(api_result.correct),
                "timed_out": bool(api_result.timed_out),
                "earned_after": api_result.earned_amount,
            })

        if question_bank is not None:
            if api_result.correct:
                question_bank.record(question.text, option_texts, chosen_idx, comp_id, level)
            elif not api_result.timed_out:
                question_bank.eliminate(question.text, option_texts, chosen_idx, comp_id, level)

        if api_result.timed_out:
            if verbose: print(f"TIMED OUT at level {level}. Final: ${api_result.earned_amount:,.2f}")
            break
        if not api_result.correct:
            if verbose: print(f"WRONG. Final: ${api_result.earned_amount:,.2f}")
            break
        if verbose:
            print(f"CORRECT. Earned so far: ${api_result.earned_amount:,.2f}")

    if verbose:
        print(f"\n=== Game over === Reached level {game.current_level}, earned ${game.earned_amount:,.2f}")
    return game

In [ ]:
import tempfile
import io
from faster_whisper import WhisperModel

def transcribe_audio(audio_bytes: bytes, language: str | None = None) -> str:
    """
    Transcribe raw WAV bytes with faster-whisper entirely in memory.
    """
    wmodel = _get_whisper()
    lang = language if language else SPEECH_LANGUAGE
    audio_stream = io.BytesIO(audio_bytes)

    try:
        segments, info = wmodel.transcribe(
            audio_stream,
            language=lang,
            beam_size=5
        )

        text = " ".join([segment.text for segment in segments])
        return text.strip()

    except Exception as e:
        print(f"[ERROR] Transcription failed: {e}")
        return ""

In [ ]:
# play one game sanity check

# Available competitions:
#   0: Entertainment
#   1: Ancient History and Politics
#   2: Science and Nature
#   3: Maths

comp_id = 0
print("\n=== Starting Game ===")
game = client.game.start(competition_id=1, mode="speech")
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")

eval_log = []
play_leaderboard_unified(model, game, question_bank=question_bank, eval_log=eval_log, verbose=True)
print(f"\nBank now contains {len(question_bank)} unique questions")


=== Starting Game ===
Session ID: 353422
Total number of questions: 15

[SYSTEM] Starting game in SPEECH mode.
[SYSTEM] Strategy: P0_baseline | RAG: False

--- Level 1 (Speech Capture) ---
  [question] RAW  : Which term in Homer's works is used to refer to the Greeks as a whole,  often accompanied by the epithet, long-haired?
  [question] CLEAN: Which term in Homer's works is used to refer to the Greeks as a whole, often accompanied by the epithet "long-haired"?
  [option_A] RAW  : Option A. Pan-L in these.
  [option_A] CLEAN: Pan-Hellenes
  [option_B] RAW  : Option B!  Ha ha ha ha ha ha ha ha ha ha ha ha ha ah, uh, uh Cains.
  [option_B] CLEAN: Achaeans
  [option_C] RAW  : Option C.  Dainans.
  [option_C] CLEAN: Danaans
  [option_D] RAW  : Option D are Gives.
  [option_D] CLEAN: Greeks
Time remaining: 9.5s
Chose option 2 -> 'Achaeans' (latency 0.93s, conf 1.00)
CORRECT. Earned so far: $100.00

--- Level 2 (Speech Capture) ---
  [question] RAW  : What was the primary reason for the ba

# Evaluation

In [ ]:
# play one game sanity check

# Available competitions:
#   0: Entertainment
#   1: Ancient History and Politics
#   2: Science and Nature
#   3: Maths
#   4: Philosophy and Psychology
#   5: News

comp_id = 0
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")

eval_log = []
play_leaderboard(model, game, eval_log=eval_log, verbose=True, use_rag=True)


=== Starting Game ===
Session ID: 354745
Total number of questions: 15

--- Level 1 ---
Q: What is the fundamental principle of Andy Dufresne's survival in Shawshank State Penitentiary?
  [1] Building a friendship with Red
  [2] Bribing the guards
  [3] Hiding his true identity
  [4] Finding a way to escape
Time remaining: 29.9s
Chose option 4 -> 'Finding a way to escape' (latency 4.84s, conf 1.00)
WRONG. Final: $0.00

=== Game over === Reached level 1, earned $0.00


In [ ]:
# online evaluation function

COMP_NAMES = {
    0: "Entertainment",
    1: "Ancient History and Politics",
    2: "Science and Nature",
    3: "Maths",
    4: "Philosophy and Psychology",
    5: "News"
}

PARTIAL_RESULTS_DIR = "/kaggle/working/partial_results"

def online_evaluate(model, client, competitions=(0, 1, 2, 3, 4, 5),
                    max_questions=50, strategy=None, use_rag=None,
                    save_partial=True, partial_dir=PARTIAL_RESULTS_DIR,
                    max_consecutive_failures=3):
    """
    Play many games against the live API. Robust on three levels:
    - per-question: retries inside play_leaderboard (transient errors)
    - per-game: catches any leak from play_leaderboard, moves to next game
    - per-competition: after `max_consecutive_failures` game-start failures,
                       skips to the next competition
    Saves the cumulative DataFrame as CSV after each competition completes.
    On KeyboardInterrupt, returns whatever was collected so far.
    """
    full_log = []
    Path(partial_dir).mkdir(parents=True, exist_ok=True)
    run_timestamp = time.strftime("%Y%m%d_%H%M%S")

    def _save_partial_csv(label):
        if not full_log:
            return
        df_partial = pd.DataFrame(full_log)
        df_partial["competition_name"] = df_partial["competition"].map(COMP_NAMES)
        save_path = Path(partial_dir) / f"online_eval_{run_timestamp}_{label}.csv"
        df_partial.to_csv(save_path, index=False)
        print(f"  ✓ Saved {len(df_partial)} rows -> {save_path.name}")

    try:
        for comp in competitions:
            comp_log_count = 0
            consecutive_failures = 0
            print(f"\n--- Starting Comp {comp} ({COMP_NAMES.get(comp, comp)}) ---")

            with tqdm(total=max_questions, desc=f"Comp {comp}", position=0) as pbar:
                while comp_log_count < max_questions:
                    # 1. Start a game (with retry)
                    try:
                        game = _retry_api(
                            lambda: client.game.start(competition_id=comp),
                            max_retries=3,
                            label=f"game.start (Comp {comp})",
                        )
                    except Exception as e:
                        consecutive_failures += 1
                        print(f"  ✗ game.start failed: {e}")
                        if consecutive_failures >= max_consecutive_failures:
                            print(f"  ✗ Too many consecutive start failures for Comp {comp}; "
                                  f"moving on")
                            break
                        time.sleep(3.0)
                        continue

                    # 2. Play the game (safety net for unexpected exceptions)
                    start_n = len(full_log)
                    try:
                        play_leaderboard(
                            model, game,
                            eval_log=full_log,
                            verbose=False,
                            strategy=strategy,
                            use_rag=use_rag,
                        )
                    except Exception as e:
                        consecutive_failures += 1
                        print(f"  ✗ play_leaderboard crashed unexpectedly: {e}")
                        if consecutive_failures >= max_consecutive_failures:
                            print(f"  ✗ Too many consecutive crashes for Comp {comp}; "
                                  f"moving on")
                            break
                        time.sleep(3.0)
                        continue

                    consecutive_failures = 0  # success resets the counter

                    added = len(full_log) - start_n
                    cap = max_questions - comp_log_count
                    if added > cap:
                        full_log = full_log[: start_n + cap]
                        added = cap
                    comp_log_count += added
                    pbar.update(added)

            # Save partial CSV after each competition completes
            if save_partial:
                _save_partial_csv(f"through_comp{comp}")

    except KeyboardInterrupt:
        print(f"\n  ⚠ Interrupted by user. Saving what we have so far.")
        if save_partial:
            _save_partial_csv("interrupted")

    df = pd.DataFrame(full_log)
    if not df.empty:
        df["competition_name"] = df["competition"].map(COMP_NAMES)
    return df


def report_results(df, model_name=MODEL_NAME):
    """
    Print accuracy / latency summaries from an evaluation DataFrame.
    """
    # Backward-compatible defaults for older logs without RAG columns
    df = df.copy()
    if "rag_latency" not in df.columns:
        df["rag_latency"] = 0.0
    if "total_latency" not in df.columns:
        df["total_latency"] = df["latency"] + df["rag_latency"]

    used_rag = df["rag_latency"].sum() > 0

    print("=" * 64)
    print(f"RESULTS — {model_name}" + (" (with RAG)" if used_rag else ""))
    print("=" * 64)

    # Per competition
    print("\nAccuracy per competition (online, biased toward early levels):")
    by_comp = df.groupby("competition_name").agg(
        n=("correct", "size"),
        acc=("correct", "mean"),
        timeouts=("timed_out", "sum"),
    )
    print(by_comp.to_string())

    # Per level
    print("\nAccuracy per level (note: smaller n at higher levels):")
    by_level = df.groupby("level").agg(
        n=("correct", "size"),
        acc=("correct", "mean"),
    )
    print(by_level.to_string())

    # Cross
    print("\nAccuracy per competition × level:")
    cross = df.pivot_table(
        index="level", columns="competition_name",
        values="correct", aggfunc="mean",
    )
    print(cross.round(3).to_string())

    # Latency total = model + retrieval (what the server actually sees)
    df = df.copy()
    if "rag_latency" not in df.columns:
        df["rag_latency"] = 0.0
    if "total_latency" not in df.columns:
        df["total_latency"] = df["latency"] + df["rag_latency"]

    print("\nLatency per competition (s):")
    lat_by_comp = df.groupby("competition_name").agg(
        n=("total_latency", "size"),
        mean_total=("total_latency", "mean"),
        mean_model=("latency", "mean"),
        mean_rag=("rag_latency", "mean"),
        max_total=("total_latency", "max"),
    )
    print(lat_by_comp.round(3).to_string())

    # Timeout danger uses TOTAL latency
    risky = (df["total_latency"] > LATENCY_DANGER_THRESHOLD_S).sum()
    print(f"\nQuestions with total latency > {LATENCY_DANGER_THRESHOLD_S}s: "
          f"{risky}/{len(df)}")

In [ ]:
df = online_evaluate(model, client, max_questions=100, use_rag=True)
report_results(df)


--- Starting Comp 0 (Entertainment) ---


Comp 0: 100%|██████████| 100/100 [23:24<00:00, 14.05s/it]


  ✓ Saved 100 rows -> online_eval_20260602_102239_through_comp0.csv

--- Starting Comp 1 (Ancient History and Politics) ---


Comp 1:  55%|█████▌    | 55/100 [10:49<08:59, 11.98s/it]

  ⚠ game.answer (Comp 1 L2) failed (attempt 1/3): Could not connect to server at http://131.175.15.22:51111: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
    Retrying in 1.0s...


...

  ✓ Saved 500 rows -> online_eval_20260602_102239_through_comp4.csv

--- Starting Comp 5 (News) ---


Comp 5:  92%|█████████▏| 92/100 [06:23<00:40,  5.10s/it]

  ⚠ game.answer (Comp 5 L5) failed (attempt 1/3): Could not connect to server at http://131.175.15.22:51111: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
    Retrying in 1.0s...


Comp 5: 100%|██████████| 100/100 [06:59<00:00,  4.20s/it]

  ✓ Saved 600 rows -> online_eval_20260602_102239_through_comp5.csv
RESULTS — Qwen/Qwen2.5-14B-Instruct (with RAG)

Accuracy per competition (online, biased toward early levels):
                                n   acc  timeouts
competition_name                                 
Ancient History and Politics  100  0.88         0
Entertainment                 100  0.85         0
Maths                         100  0.66         6
News                          100  0.83         0
Philosophy and Psychology     100  0.97         0
Science and Nature            100  0.90         0

Accuracy per level (note: smaller n at higher levels):
         n       acc
level               
1      103  0.786408
2       81  0.888889
3       72  0.805556
4       58  0.879310
5       50  0.920000
6       46  0.869565
7       39  0.846154
8       33  0.818182
9       26  0.769231
10      20  0.950000
11      18  0.944444
12      17  0.882353
13      15  0.866667
14      12  0.833333
15      10  0.700000

Accurac

# Prompt Experiment

This is a separate experiment to analyze how different prompting approaches affect performance.

In [ ]:
def online_prompt_ablation(model, client, strategies, max_questions=30,
                           competitions=(0, 1, 2, 3, 4, 5), use_rag=None):
    """
    Compare prompt strategies by running each online against the live API.

    All strategies see fresh, independent samples from the question pool — no
    bank, no offline mode. Each (strategy, competition) cell gets max_questions
    questions logged.

    Args:
        strategies: list of PromptStrategy objects to compare
        use_rag: optional override applied to ALL strategies. If None, each
                 strategy's own .use_rag field is used.
    """
    all_results = {}

    for strategy in strategies:
        print(f"\n{'=' * 64}\nSTRATEGY — {strategy.name}\n{'=' * 64}")
        df = online_evaluate(
            model, client,
            max_questions=max_questions,
            competitions=competitions,
            strategy=strategy,
            use_rag=use_rag,
        )
        df["strategy"] = strategy.name
        all_results[strategy.name] = df

    return pd.concat(all_results.values(), ignore_index=True)


def report_prompt_ablation(df):
    print("\n" + "=" * 64)
    print("PROMPT ABLATION RESULTS (online)")
    print("=" * 64)

    print("\nOverall accuracy by strategy:")
    print(df.groupby("strategy")["correct"].agg(
        n="count", accuracy="mean"
    ).round(3).to_string())

    print("\nAccuracy by competition × strategy:")
    print(df.pivot_table(
        index="competition_name", columns="strategy",
        values="correct", aggfunc="mean"
    ).round(3).to_string())

    print("\nModel latency (s) by strategy:")
    print(df.groupby("strategy")["latency"].agg(
        mean="mean", p95=lambda x: x.quantile(0.95), max="max"
    ).round(3).to_string())

    if "rag_latency" in df.columns and df["rag_latency"].sum() > 0:
        df_local = df.copy()
        df_local["total_latency"] = df_local["latency"] + df_local["rag_latency"]
        print("\nTotal latency (s) by strategy (model + retrieval):")
        print(df_local.groupby("strategy")["total_latency"].agg(
            mean="mean", p95=lambda x: x.quantile(0.95), max="max"
        ).round(3).to_string())

    print("\nTimeouts by strategy:")
    print(df.groupby("strategy")["timed_out"].sum().to_string())

    print("\nCalibration: mean confidence on correct vs incorrect:")
    print(df.groupby(["strategy", "correct"])["confidence"]
          .apply(lambda s: s.apply(
              lambda c: c.get(max(c, key=c.get), 0) if isinstance(c, dict) else c
          ).mean())
          .round(3).to_string())

In [ ]:
RUN_THIS_CELL = False  # change to True to run prompt experiment

if RUN_THIS_CELL:
    # Run prompt experiment
    df_prompt_ab = online_prompt_ablation(
        model, client,
        strategies=[P1_FEWSHOT, P4_SPECIALIZED, P5_ELIMINATE, P2_COT],
        max_questions=100, # per competition
    )
    report_prompt_ablation(df_prompt_ab)


STRATEGY — P0_baseline

--- Starting Comp 0 (Entertainment) ---


Comp 0: 100%|██████████| 30/30 [00:25<00:00,  1.18it/s]


  ✓ Saved 30 rows -> online_eval_20260526_151106_through_comp0.csv

--- Starting Comp 1 (Ancient History and Politics) ---


Comp 1: 100%|██████████| 30/30 [00:29<00:00,  1.00it/s]


  ✓ Saved 60 rows -> online_eval_20260526_151106_through_comp1.csv

--- Starting Comp 2 (Science and Nature) ---


...

  ✓ Saved 120 rows -> online_eval_20260526_170754_through_comp3.csv

--- Starting Comp 4 (Philosophy and Psychology) ---


Comp 4: 100%|██████████| 30/30 [06:33<00:00, 13.11s/it]


  ✓ Saved 150 rows -> online_eval_20260526_170754_through_comp4.csv

--- Starting Comp 5 (News) ---


Comp 5: 100%|██████████| 30/30 [05:37<00:00, 11.24s/it]

  ✓ Saved 180 rows -> online_eval_20260526_170754_through_comp5.csv

PROMPT ABLATION RESULTS (online)

Overall accuracy by strategy:
                       n  accuracy
strategy                          
P0_baseline          180     0.789
P1_fewshot           180     0.756
P2_cot               180     0.800
P3_self_consistency  180     0.739
P4_specialized       180     0.778
P5_eliminate         180     0.794

Accuracy by competition × strategy:
strategy                      P0_baseline  P1_fewshot  P2_cot  P3_self_consistency  P4_specialized  P5_eliminate
competition_name                                                                                                
Ancient History and Politics        0.833       0.933   0.900                0.867           0.867         0.833
Entertainment                       0.833       0.700   0.733                0.800           0.933         0.833
Maths                               0.600       0.533   0.700                0.333           0.333

# RAG Experiment

The investigation itself is in the report: the dropped sources, the stack of *silent*
failures that each invalidated weeks of measurement (capped API params, disambiguation
leaks, an undefined `HEADERS` swallowed by a broad `except`, a parquet row-group
indexing bug that fetched empty text for canonical articles), the News win, and the
finding that feeding whole articles beats cross-encoder reranking.

Two artifacts remain:
- **`paired_rag_diagnostic`** — same-question RAG vs no-RAG on the live game. The
  server never reveals the gold answer, so a paired run (ask both ways, submit RAG's
  pick) is the only valid comparison. Prints, per wrong question, what the routed
  retriever actually surfaced.
- **`rag_vs_norag_eval`** — runs the RAG categories twice in one call (with / without
  RAG) on independent games and tabulates accuracy, latency, and timeouts side by side
  (equivalent to two `online_evaluate` runs, reported together).


In [ ]:
RUN_THIS_CELL = False   # set True to run RAG vs NO-RAG experiment

def rag_vs_norag_eval(model, client, competitions=(0, 1), max_questions=100):
    """Run the RAG categories twice in one call -- once WITH rag, once WITHOUT --
    on independent games, and report a side-by-side accuracy / latency / timeout
    comparison. Equivalent to running online_evaluate twice, reported together."""
    print("=== arm 1/2: WITH rag ===")
    df_rag   = online_evaluate(model, client, competitions=competitions,
                               max_questions=max_questions, use_rag=True, save_partial=False)
    print("\n=== arm 2/2: WITHOUT rag ===")
    df_norag = online_evaluate(model, client, competitions=competitions,
                               max_questions=max_questions, use_rag=False, save_partial=False)
    df_rag["arm"] = "rag"; df_norag["arm"] = "no_rag"
    both = pd.concat([df_rag, df_norag], ignore_index=True)

    acc = both.pivot_table(index="competition_name", columns="arm",
                           values="correct", aggfunc="mean")
    if "rag" in acc and "no_rag" in acc:
        acc["delta"] = acc["rag"] - acc["no_rag"]
    n = both.pivot_table(index="competition_name", columns="arm",
                         values="correct", aggfunc="size")
    if "total_latency" not in both.columns:
        both["total_latency"] = both.get("latency", 0) + both.get("rag_latency", 0)
    lat = both.groupby("arm").agg(mean_total=("total_latency", "mean"),
                                  max_total=("total_latency", "max"),
                                  timeouts=("timed_out", "sum"))

    print("\n" + "=" * 60)
    print("RAG vs no-RAG  (independent games per arm)")
    print("=" * 60)
    print("\nAccuracy:");           print(acc.round(3).to_string())
    print("\nQuestions per arm:");  print(n.to_string())
    print("\nLatency / timeouts:"); print(lat.round(2).to_string())
    return df_rag, df_norag

if RUN_THIS_CELL:
    df_rag, df_norag = rag_vs_norag_eval(model, client, competitions=(0, 1), max_questions=100)

In [ ]:
RUN_THIS_CELL = False   # set True to run RAG diagnostic experiment

import time, gc, pandas as pd, torch

def _api(fn, *args, retries=5, base_sleep=0.6, **kwargs):
    _CONN = ("RemoteDisconnected", "Connection aborted", "Connection reset",
             "ConnectionResetError", "ProtocolError", "closed connection",
             "ConnectionError", "Max retries exceeded")
    last = None
    for attempt in range(retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if not any(s in repr(e) for s in _CONN):
                raise
            last = e
            time.sleep(base_sleep * (attempt + 1))
    raise last

def _free_cuda():
    """Reclaim GPU memory across all devices after an OOM."""
    gc.collect()
    if torch.cuda.is_available():
        for d in range(torch.cuda.device_count()):
            with torch.cuda.device(d):
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()

def predict(qtext, opts, comp_id, use_rag):
    res  = ask_llm(model, qtext, opts, comp_id, strategy=None, use_rag=use_rag)
    conf = res["confidence"] if isinstance(res, dict) else getattr(res, "confidence", None)
    if conf is None and isinstance(res, (tuple, list)):
        conf = next((x for x in res if isinstance(x, dict)), None)
    best = max(conf, key=conf.get)
    return int(best) - 1, conf

def safe_predict(qtext, opts, comp_id, use_rag, retries=1):
    """predict() but tolerant of CUDA OOM: free memory, retry once, then give up
    (returns None) so the caller can skip the question instead of crashing."""
    for attempt in range(retries + 1):
        try:
            return predict(qtext, opts, comp_id, use_rag)
        except torch.cuda.OutOfMemoryError:
            _free_cuda()
        except RuntimeError as e:
            if "out of memory" not in str(e).lower():
                raise
            _free_cuda()
        if attempt < retries:
            time.sleep(0.5)   # let the cache settle before retrying
    return None

def paired_rag_diagnostic(competitions=(0, 1), n_per_comp=60, show_wrong=5):
    rows, wrong = [], []
    oom_skips = 0
    for comp in competitions:
        collected = 0
        while collected < n_per_comp:
            game = _api(client.game.start, competition_id=comp)
            while game.in_progress and collected < n_per_comp:
                q = game.current_question
                if q is None:
                    break
                opts = [o.text for o in q.options]
                t0 = time.time()
                rag = safe_predict(q.text, opts, comp, use_rag=True)
                rag_lat = time.time() - t0
                norag = safe_predict(q.text, opts, comp, use_rag=False)

                if rag is None or norag is None:        # OOM on this question
                    oom_skips += 1
                    _free_cuda()
                    print(f"  [skip] OOM on comp{comp} question; abandoning game "
                          f"(total skipped: {oom_skips})")
                    break                                # start a fresh game

                rag_idx, rag_conf = rag
                norag_idx, _      = norag
                ans = _api(game.answer, q.options[rag_idx].id)   # submit RAG's pick
                rag_correct = bool(ans.correct)
                differ = (rag_idx != norag_idx)
                rows.append({"comp": comp, "rag_correct": rag_correct,
                             "differ": differ, "rag_latency": rag_lat})
                if not rag_correct:
                    active = routing_retriever.routes.get(comp, routing_retriever.default)
                    dbg = (active.retrieve_debug(q.text, opts)
                           if active is not None and hasattr(active, "retrieve_debug")
                           else {"articles": [], "top": []})
                    wrong.append({"comp": comp, "q": q.text,
                                  "rag_pick": opts[rag_idx], "norag_pick": opts[norag_idx],
                                  "conf": rag_conf, "articles": dbg["articles"], "top": dbg["top"]})
                collected += 1
                if ans.timed_out or not ans.correct:
                    break

    df = pd.DataFrame(rows)
    acc = df["rag_correct"].mean()
    print(f"collected {len(df)} | OOM-skipped {oom_skips} | RAG acc {acc:.3f} | "
          f"rag_latency median {df['rag_latency'].median():.2f}s max {df['rag_latency'].max():.2f}s")
    print(df.groupby("comp")["rag_correct"].mean(), "\n")

    diff = df[df["differ"]]
    pure_gain  = int((diff["rag_correct"]).sum())
    maybe_hurt = int((~diff["rag_correct"]).sum())
    print(f"RAG changed the answer on {len(diff)}/{len(df)} questions:")
    print(f"  PURE GAIN  (differ & RAG correct -> no_rag was wrong): {pure_gain:3d}")
    print(f"  MAYBE HURT (differ & RAG wrong  -> no_rag maybe right): {maybe_hurt:3d}")
    print(f"  agreed on the other {len(df) - len(diff)}")
    print(f"  net benefit is between {-maybe_hurt} (worst) and {pure_gain} (best)\n")

    print("=== WRONG answers: articles + what retrieval surfaced ===")
    for w in wrong[:show_wrong]:
        print(f"\n[wrong] comp{w['comp']} conf={w['conf']}")
        print(f"  Q: {w['q']}")
        print(f"  RAG picked:    {w['rag_pick']!r}")
        print(f"  no_rag picked: {w['norag_pick']!r}"
              + ("  (same)" if w['rag_pick'] == w['norag_pick'] else "  (DIFFERENT)"))
        print(f"  articles: {w['articles']}")
        for score, passage in w["top"]:
            print(f"    [{score:+.2f}] {passage}")
    return df, wrong

if RUN_THIS_CELL:
    diag_df, diag_wrong = paired_rag_diagnostic(competitions=(0, 1), n_per_comp=50, show_wrong=10)

collected 120 | RAG acc 0.892 | rag_latency median 8.27s max 18.53s
comp
0    0.850000
1    0.933333
Name: rag_correct, dtype: float64 

RAG changed the answer on 12/120 questions:
  PURE GAIN  (differ & RAG correct -> no_rag was wrong):   9
  MAYBE HURT (differ & RAG wrong  -> no_rag maybe right):   3
  agreed on the other 108
  net benefit is between -3 (worst) and 9 (best)

=== WRONG answers: articles + what retrieval surfaced ===

[wrong] comp0 conf={1: 5.0334898817913345e-09, 2: 8.833895334076059e-06, 3: 1.9789644741090023e-08, 4: 0.9999911412815313}
  Q: How does the film 'The Shawshank Redemption' primarily convey its message of hope?
  RAG picked:    "By Red's final speech"
  no_rag picked: "By Red's final speech"  (same)
  articles: ['The Shawshank Redemption']
    [+0.00] The Shawshank Redemption is a 1994 American drama film written and directed by Frank Darabont, based on the 19

[wrong] comp0 conf={1: 0.9999995858159111, 2: 2.187566670655485e-09, 3: 5.379543381266311e-11, 

# Results: Unoptimized vs Optimized Performance

**Unoptimized** = no RAG, no CoT (plain `ask_llm` baseline for every category).

**Optimized** = Wikipedia RAG for Ancient History, Entertainment and Science;
Google-News RAG for News; chain-of-thought prompting for Maths; Philosophy left
unchanged (RAG showed no benefit).

### Accuracy

| Competition | Optimization | Unoptimized | Optimized | Δ |
|---|---|---:|---:|---:|
| Entertainment | Wikipedia RAG | 0.822 | 0.850 | +0.028 |
| Ancient History & Politics | Wikipedia RAG | 0.846 | 0.892 | +0.046 |
| Science & Nature | Wikipedia RAG | 0.910 | 0.930 | +0.020 |
| Maths | Chain-of-thought | 0.624 | 0.750 | +0.126 |
| Philosophy & Psychology | none (no change) | 0.962 | 0.962 | 0.000 |
| News | Google-News RAG | 0.592 | 0.826 | +0.234 |
| **Overall (mean)** | — | **0.793** | **0.868** | **+0.076** |

### Latency (total seconds: model + retrieval — what the 30 s server clock sees)

| Competition | Unopt. mean | Unopt. max | Opt. mean | Opt. max |
|---|---:|---:|---:|---:|
| Entertainment | 0.82 | 0.92 | 14.31 | 26.73 |
| Ancient History & Politics | 0.81 | 0.88 | 12.02 | 26.07 |
| Science & Nature | 0.80 | 0.89 | 12.37 | 27.40 |
| Maths | 0.85 | 1.44 | 7.96 | 21.91 |
| Philosophy & Psychology | 0.82 | 0.87 | 0.82 | 0.87 |
| News | 0.82 | 0.85 | 4.22 | 14.73 |

### Reading the results

- **News is the largest single gain** (+23.4 pp, 0.592 → 0.826): Google-News
  retrieval supplies the recency the base model lacks. It is also the cheapest
  optimized category (4.2 s mean) because news bodies are short.
- **Maths gains +12.6 pp** (0.624 → 0.750) from chain-of-thought alone — no tools,
  no retrieval — at the cost of latency rising from ~0.9 s to ~8 s.
- **Knowledge categories** (Entertainment, Ancient History, Science) gain modestly
  (+2 to +5 pp); they were already near ceiling, so retrieval adds little.
- **Philosophy is left unchanged**: RAG produced no measurable benefit, so the
  no-RAG path is retained — identical accuracy and latency in both arms.
- **Cost of optimization**: retrieval/CoT push mean latency from under 1 s to
  ~12–14 s and peak latency to ~27 s (Science), still inside the 30 s budget with
  zero timeouts, but the RAG categories run close to the wall.